In [ ]:
import os.path
import re
import sys

sys.path.extend([".", ".."])
import time
from tqdm import tqdm
import json
import cProfile

# from transformers import AutoTokenizer
from utils.java_parser import parse_fields_from_class_code
from data.configuration import code_base
from utils.prompt_formats.new_prompt_formatter import PromptFormatter

In [ ]:
profiler = cProfile.Profile()
focal_method_key = "source:source_method_code_format"
focal_method_name_key = "source:source_method_name"
focal_method_signature_key = "source:source_method_signature"
focal_method_docstring_key = "source:source_method_docstring"
focal_method_param_key = "source:source_method_parameters"
focal_method_return_type_key = "source:source_return_type"
focal_class_other_methods_key = "source:source_other_method_signature"
focal_class_fields_key = "source_class_fields"
focal_class_constructor_key = "content:source_class_constructors"
focal_class_name_key = "content:source_class_name"
focal_class_code_key = "content:source_class_code_format"
focal_class_docstring_key = "content:source_class_docstring"
source_method_type_constructor_key = "content:parameter_class_constructors"
source_method_return_type_constructor_key = "content:return_type_class_constructors"
source_method_parameter_key = "content:parameter_list"
parameter_classes_key = "content:parameter_class_signature"
source_method_signature_key = "source:source_method_signature"
source_class_imports_key = "content:source_class_code_imports"
project_name_key = 'extra:project_name'

In [ ]:
formatter = PromptFormatter()

In [ ]:
model_name = 'chatgpt'
tgt_model = 'Claude-4-Sonnet'
format = 'comment'
strategy = 'generation'
prompt_version = 'buggy'
test_log_version = 'buggy'
ignore_feature = "full"
few_shot_examples = 5
min_test_num = 10

In [ ]:
file_post_fix = f'unified_invoked'
# file_post_fix = f'unified_invoked_code_prompt'
# file_post_fix = f'unified_invoked_docstring_prompt'
# file_post_fix = f'unified_invoked_docstring_code_prompt'
# file_post_fix = f'unified_invoked_docstring_cot_prompt'
# file_post_fix = f'unified_invoked_docstring_cot_prompt_min_test_num_{min_test_num}'
# file_post_fix = f'unified_invoked_docstring_cot_prompt_min_test_num_{min_test_num}_err_msg_prompt'
# file_post_fix = f'unified_invoked_docstring_cot_few_shot_prompt'
# file_post_fix = f'unified_invoked_code_cot_few_shot_prompt_min_test_num_{min_test_num}'
# file_post_fix = f'unified_invoked_docstring_cot_few_shot_prompt_min_test_num_{min_test_num}'
# file_post_fix = f'unified_invoked_docstring_cot_few_shot_example_num_{few_shot_examples}_prompt_min_test_num_{min_test_num}'

In [ ]:
# Load the data
data = []
source_data_file = os.path.join(code_base, f"data/prompts/{tgt_model}_{prompt_version}_source_data_unified_invoked_with_docstring.jsonl")
if os.path.exists(source_data_file):
    with open(source_data_file, "r") as f:
        for line in f:
            data.append(json.loads(line))

In [ ]:
test_log = []
test_log_file = f'/path/to/repo/data/rq1/test_log/{tgt_model}_{format}_{strategy}_{ignore_feature}_{test_log_version}_{file_post_fix}_test_logs.jsonl'
if os.path.exists(test_log_file):
    with open(test_log_file, "r") as f:
        for line in f:
            test_log.append(json.loads(line))

In [ ]:
test_log_file

In [ ]:
len(data)

In [ ]:
example_idx = 0

In [ ]:
example_instance = data[example_idx]

In [ ]:
example_test_log_instance = test_log[example_idx]

In [ ]:
example_test_log_instance

In [ ]:
example_test_log_instance['method_signature']

In [ ]:
assert example_instance['source:source_method_signature'] == example_test_log_instance['method_signature']

In [ ]:
print(example_instance[focal_method_key])

In [ ]:
example_instance[focal_method_return_type_key]

In [ ]:
example_new_inst = {}
bug_id = example_instance["extra:project_name"]
example_new_inst["focal_method_invoked"] = example_instance["focal_method_invoked"]
example_new_inst["id"] = bug_id
example_new_inst["strategy"] = strategy
example_new_inst["format"] = format
example_new_inst["ablation"] = ignore_feature
example_new_inst["focal_method"] = example_instance[focal_method_key]
is_public, example_new_inst['content'] = formatter.apply_format_with_docstring(
    example_instance,
    model_name,
    strategy=strategy,
    formatting=format,
    ignore_feature=ignore_feature,
    project_name=bug_id,
    # min_test_num=min_test_num,
    # with_examples=True,
    with_docstring=False,
    with_code=True,
    # with_generated_focal=True,
    # with_cot=True,
    # few_shot_examples = few_shot_examples,
    # force_no_triggered=True,
    for_seq_score=True
)
example_new_inst["class_name"] = bug_id
example_new_inst["method_signature"] = example_instance['source:source_method_signature']
example_new_inst["is_public"] = "1" if is_public else "0"
example_new_inst["role"] = "user"


In [ ]:
print(example_new_inst['content'])

In [ ]:
# print(example_test_log_instance['ut_logs']['effective_uts'][0]['method_text'])

In [ ]:
if example_test_log_instance['ut_logs']['effective_uts']:
    example_effective_json_instance = {"prefix": example_new_inst['content'],
                              "responses": [effective_ut['method_text'] for effective_ut in example_test_log_instance['ut_logs']['effective_uts']]}
    print(example_effective_json_instance)


In [ ]:
if example_test_log_instance['ut_logs']['misguided_uts']:
    example_misguided_json_instance = {"prefix": example_new_inst['content'],
                              "responses": [misguided_ut['method_text'] for misguided_ut in example_test_log_instance['ut_logs']['misguided_uts']]}
    for idx, i in enumerate(example_misguided_json_instance['responses']):
        print(f"Misguided UT {idx}: {i}")


In [ ]:
effective_ut_json = []
misguided_ut_json = []

In [ ]:
pbar = tqdm(data, total=len(data))
for idx, instance in enumerate(pbar):
    new_instance = {}
    bug_id = instance["extra:project_name"]
    new_instance["focal_method_invoked"] = instance["focal_method_invoked"]
    new_instance["id"] = bug_id
    new_instance["strategy"] = strategy
    new_instance["format"] = format
    new_instance["ablation"] = ignore_feature
    new_instance["focal_method"] = instance[focal_method_key]
    is_public, new_instance['content'] = formatter.apply_format_with_docstring(
        instance,
        model_name,
        strategy=strategy,
        formatting=format,
        ignore_feature=ignore_feature,
        project_name=bug_id,
        # min_test_num=min_test_num,
        # with_examples=True,
        with_docstring=False,
        with_code=True,
        # with_generated_focal=True,
        # with_cot=True,
        # few_shot_examples = few_shot_examples,
        # force_no_triggered=True,
        for_seq_score=True
    )
    new_instance["class_name"] = bug_id
    new_instance["method_signature"] = instance['source:source_method_signature']
    new_instance["is_public"] = "1" if is_public else "0"
    new_instance["role"] = "user"
    new_test_log_instance = test_log[idx]
    assert new_test_log_instance['method_signature'] == instance['source:source_method_signature']
    if new_test_log_instance['ut_logs']['effective_uts']:
        new_effective_json_instance = {"prefix": new_instance['content'],
                                       "responses": [effective_ut['method_text'] for effective_ut in new_test_log_instance['ut_logs']['effective_uts']]}
        effective_ut_json.append(new_effective_json_instance)
    if new_test_log_instance['ut_logs']['misguided_uts']:
        new_misguided_json_instance = {"prefix": new_instance['content'],
                                       "responses": [effective_ut['method_text'] for effective_ut in new_test_log_instance['ut_logs']['misguided_uts']]}
        misguided_ut_json.append(new_misguided_json_instance)

In [ ]:
len([e for e in effective_ut_json])

In [ ]:
sum([len(e['responses']) for e in effective_ut_json])

In [ ]:
len([m for m in misguided_ut_json])

In [ ]:
sum([len(m['responses']) for m in misguided_ut_json])

In [ ]:
effective_ut_file = os.path.join(
    code_base,
    "/path/to/repo/data/seq_score_calc/"
    + tgt_model
    + f"_{format}_{strategy}_{ignore_feature}_{prompt_version}_{file_post_fix}_prompt_{test_log_version}_generated_effective_uts_pair.json",
    # + f"_{format}_{strategy}_{ignore_feature}_buggy_unified_invoked_docstring_cot_prompt.jsonl",
)

In [ ]:
effective_ut_file

In [ ]:
# dump effective_ut_json to effective_ut_file
with open(effective_ut_file, "w") as f:
    json.dump(effective_ut_json, f, indent=4)

In [ ]:
# read the effective_ut_json to see if the data is written correctly
with open(effective_ut_file, "r") as f:
    written_effective_ut_data = json.load(f)

In [ ]:
len(written_effective_ut_data)

In [ ]:
print(written_effective_ut_data[0]['prefix'])

In [ ]:
len([e for e in written_effective_ut_data if e['responses']])

In [ ]:
sum([len(e['responses']) for e in written_effective_ut_data])

In [ ]:
misguided_ut_file = os.path.join(
    code_base,
    "/path/to/repo/data/seq_score_calc/"
    + tgt_model
    + f"_{format}_{strategy}_{ignore_feature}_{prompt_version}_{file_post_fix}_prompt_{test_log_version}_generated_misguided_uts_pair.json",
    # + f"_{format}_{strategy}_{ignore_feature}_buggy_unified_invoked_docstring_cot_prompt.jsonl",
)

In [ ]:
misguided_ut_file

In [ ]:
# dump misguided_ut_json to misguided_ut_file
with open(misguided_ut_file, "w") as f:
    json.dump(misguided_ut_json, f, indent=4)


In [ ]:
# read the misguided_ut_json to see if the data is written correctly
with open(misguided_ut_file, "r") as f:
    written_misguided_ut_data = json.load(f)


In [ ]:
len(written_misguided_ut_data)
print(written_misguided_ut_data[0]['prefix'])

In [ ]:
len([m for m in written_misguided_ut_data if m['responses']])

In [ ]:
sum([len(m['responses']) for m in written_misguided_ut_data])

In [ ]:
models = ['DeepSeek-R1', 'GPT-O4-MINI', 'Gemini-2-5-flash-thinking', 'Gemini-2-5-Pro',  'Grok-4', 'Qwen-3-Plus', 'Claude-4-Sonnet-extended-thinking', 'GPT-4-1', 'Claude-4-Sonnet', 'Gemini-2-5-flash', 'DeepSeek-V3', 'Grok-3', 'Qwen-3-Coder-Plus']
# models = ['Gemini-2-5-flash-thinking']


In [ ]:
version = ['fixed', 'buggy']

In [ ]:
success_models = []

In [ ]:
for tgt_model in models:
    print(f"processing model: {tgt_model}")
    missing_file = False
    for prompt_version in version:
        for test_log_version in version:
            source_data_file = os.path.join(code_base, f"data/prompts/{tgt_model}_{prompt_version}_source_data_unified_invoked_with_docstring.jsonl")
            test_log_file = f'/path/to/repo/data/rq1/test_log/{tgt_model}_{format}_{strategy}_{ignore_feature}_{test_log_version}_{file_post_fix}_test_logs.jsonl'
            # check if both file exist, if not, print model name and test log version, then continue
            if not os.path.exists(source_data_file):
                print(f"Missing source data file for model: {tgt_model}, prompt_version: {prompt_version}")
                missing_file = True
                continue
            if not os.path.exists(test_log_file):
                print(f"Missing test log file for model: {tgt_model}, prompt_version: {prompt_version}, test_log_version: {test_log_version}")
                missing_file = True
                continue
            # load the data
            data = []
            with open(source_data_file, "r") as f:
                for line in f:
                    data.append(json.loads(line))
            test_log = []
            with open(test_log_file, "r") as f:
                for line in f:
                    test_log.append(json.loads(line))
            # check if the length of data and test_log is the same
            if len(data) != len(test_log):
                print(f"Length mismatch for model: {tgt_model}, prompt_version: {prompt_version}, test_log_version: {test_log_version}")
                missing_file = True
                continue
            # process the data
            effective_ut_json = []
            misguided_ut_json = []
            pbar = tqdm(data, total=len(data))
            for idx, instance in enumerate(pbar):
                new_instance = {}
                bug_id = instance["extra:project_name"]
                new_instance["focal_method_invoked"] = instance["focal_method_invoked"]
                new_instance["id"] = bug_id
                new_instance["strategy"] = strategy
                new_instance["format"] = format
                new_instance["ablation"] = ignore_feature
                new_instance["focal_method"] = instance[focal_method_key]
                is_public, new_instance['content'] = formatter.apply_format_with_docstring(
                    instance,
                    model_name,
                    strategy=strategy,
                    formatting=format,
                    ignore_feature=ignore_feature,
                    project_name=bug_id,
                    # min_test_num=min_test_num,
                    # with_examples=True,
                    with_docstring=False,
                    with_code=True,
                    # with_generated_focal=True,
                    # with_cot=True,
                    # few_shot_examples = few_shot_examples,
                    # force_no_triggered=True,
                    for_seq_score=True
                )
                new_instance["class_name"] = bug_id
                new_instance["method_signature"] = instance['source:source_method_signature']
                new_instance["is_public"] = "1" if is_public else "0"
                new_instance["role"] = "user"
                new_test_log_instance = test_log[idx]
                assert new_test_log_instance['method_signature'] == instance['source:source_method_signature']
                if new_test_log_instance['ut_logs']['effective_uts']:
                    new_effective_json_instance = {"prefix": new_instance['content'],
                                                   "responses": [effective_ut['method_text'] for effective_ut in
                                                                 new_test_log_instance['ut_logs']['effective_uts']]}
                    effective_ut_json.append(new_effective_json_instance)
                if new_test_log_instance['ut_logs']['misguided_uts']:
                    new_misguided_json_instance = {"prefix": new_instance['content'],
                                                   "responses": [misguided_ut['method_text'] for misguided_ut in
                                                                 new_test_log_instance['ut_logs']['misguided_uts']]}
                    misguided_ut_json.append(new_misguided_json_instance)
            # print total number effective uts and misguided uts
            print(f"Model: {tgt_model}, Prompt Version: {prompt_version}, Test Log Version: {test_log_version}")
            print(f"Total Effective UTs: {len(effective_ut_json)}, Total Misguided UTs: {len(misguided_ut_json)}")
            print(f"Total Effective UTs Responses: {sum([len(e['responses']) for e in effective_ut_json])}, "
                  f"Total Misguided UTs Responses: {sum([len(m['responses']) for m in misguided_ut_json])}")
            # dump effective_ut_json to effective_ut_file
            effective_ut_file = os.path.join(
                code_base,
                f"/path/to/repo/data/seq_score_calc/"
                + tgt_model
                + f"_{format}_{strategy}_{ignore_feature}_{prompt_version}_{file_post_fix   }_prompt_{test_log_version}_generated_effective_uts_pair.json",
            )
            with open(effective_ut_file, "w") as f:
                json.dump(effective_ut_json, f, indent=4)

            # dump misguided_ut_json to misguided_ut_file
            misguided_ut_file = os.path.join(
                code_base,
                f"/path/to/repo/data/seq_score_calc/"
                + tgt_model
                + f"_{format}_{strategy}_{ignore_feature}_{prompt_version}_{file_post_fix}_prompt_{test_log_version}_generated_misguided_uts_pair.json",
            )
            with open(misguided_ut_file, "w") as f:
                json.dump(misguided_ut_json, f, indent=4)
            print(f"Effective UTs and Misguided UTs data saved to {effective_ut_file} and {misguided_ut_file}")
    if not missing_file:
        print(f"Successfully processed model: {tgt_model}, prompt_version: {prompt_version}, test_log_version: {test_log_version}")
        # append the model to success_models
        success_models.append(tgt_model)





In [ ]:
for name in success_models:
    print(name)